In [25]:
import os
import pandas as pd
import xarray as xr
import xarray as xr
import numpy as np

In [18]:
VAULT_DIR = "/data0/skagit_met/data_transfer/data"
PRISM_DIR = os.path.join(VAULT_DIR, "PRISM")
PNNL_DIR = os.path.join(VAULT_DIR, "PNNL/historical")
DAYMET_DIR = os.path.join(VAULT_DIR, "DaymetV4")
# CONUS_DIR = os.path.join(VAULT_DIR, "CONUS")
UCLA_DIR = os.path.join(VAULT_DIR, "ucla_era5_d02_daily/prec")
GRIDMET_DIR = os.path.join(VAULT_DIR, "gridmet")

# PRISM

In [3]:
# Scan all decade directories for zarr files
decade_folders = [d for d in os.listdir(PRISM_DIR) if os.path.isdir(os.path.join(PRISM_DIR, d))]
decade_folders.sort()

print("Scanning PRISM zarr files for data coverage...")
print("="*80)

coverage_by_year = {}
all_years = []

for decade in decade_folders:
    decade_path = os.path.join(PRISM_DIR, decade)
    zarr_files = [f for f in os.listdir(decade_path) if f.endswith('.zarr') and 'daily_4km' in f]

    print(f"\nDecade: {decade}")

    for zarr_file in sorted(zarr_files):
        zarr_path = os.path.join(decade_path, zarr_file)

        try:
            ds = xr.open_zarr(zarr_path, consolidated=False)
            dates = pd.to_datetime(ds.time.values)

            year = int(zarr_file.split('-')[0])
            start_date = dates.min()
            end_date = dates.max()
            expected_dates = pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31), freq='D')

            available_count = len(dates)
            expected_count = len(expected_dates)
            coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

            # Find missing dates
            missing_dates = [d for d in expected_dates if d not in dates.values]

            coverage_by_year[year] = {
                'decade': decade,
                'available': available_count,
                'expected': expected_count,
                'coverage_pct': coverage_pct,
                'start_date': start_date,
                'end_date': end_date,
                'missing_dates': missing_dates,
                'first_date': start_date,
                'last_date': end_date
            }

            all_years.append(year)

            if coverage_pct == 100:
                status = "✓ COMPLETE"
            else:
                status = f"✗ {available_count}/{expected_count} ({coverage_pct:.1f}%)"

            print(f"  {year}: {status} [{start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}]")

            ds.close()

        except Exception as e:
            print(f"  Error reading {zarr_file}: {e}")

all_years.sort()

# Print detailed summary
print("\n" + "="*80)
print("COVERAGE SUMMARY")
print("="*80)

complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

print(f"Complete coverage (100%): {complete} years")
print(f"Partial coverage (1-99%): {partial} years")
print(f"No coverage (0%): {missing} years")
print(f"\nYears available: {all_years[0]} to {all_years[-1]} ({len(all_years)} years total)")

# Show years with incomplete coverage
incomplete = [(y, coverage_by_year[y]) for y in all_years if coverage_by_year[y]['coverage_pct'] < 100]
if incomplete:
    print(f"\n" + "="*80)
    print("YEARS WITH INCOMPLETE COVERAGE")
    print("="*80)
    for year, data in incomplete:
        print(f"\n{year}: {data['coverage_pct']:.1f}% ({data['available']}/{data['expected']} dates)")
        print(f"  Range: {data['first_date'].strftime('%Y-%m-%d')} to {data['last_date'].strftime('%Y-%m-%d')}")
        if len(data['missing_dates']) <= 20:
            print(f"  Missing dates ({len(data['missing_dates'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_dates']])}")
        else:
            missing_by_month = {}
            for d in data['missing_dates']:
                key = d.strftime('%Y-%m')
                if key not in missing_by_month:
                    missing_by_month[key] = []
                missing_by_month[key].append(d)

            print(f"  Missing dates ({len(data['missing_dates'])}): Gaps by month:")
            for month_key in sorted(missing_by_month.keys()):
                dates_in_month = missing_by_month[month_key]
                print(f"    {month_key}: {len(dates_in_month)} missing days")

Scanning PRISM zarr files for data coverage...

Decade: 1981-1990
  1981: ✗ 364/365 (99.7%) [1981-01-01 to 1981-12-31]
  1982: ✓ COMPLETE [1982-01-01 to 1982-12-31]
  1983: ✓ COMPLETE [1983-01-01 to 1983-12-31]
  1984: ✗ 364/366 (99.5%) [1984-01-01 to 1984-12-31]
  1985: ✓ COMPLETE [1985-01-01 to 1985-12-31]
  1986: ✓ COMPLETE [1986-01-01 to 1986-12-31]
  1987: ✗ 360/365 (98.6%) [1987-01-01 to 1987-12-31]
  1988: ✓ COMPLETE [1988-01-01 to 1988-12-31]
  1989: ✗ 363/365 (99.5%) [1989-01-01 to 1989-12-31]
  1990: ✗ 361/365 (98.9%) [1990-01-01 to 1990-12-31]

Decade: 1991-2000
  1991: ✗ 363/365 (99.5%) [1991-01-01 to 1991-12-31]
  1992: ✓ COMPLETE [1992-01-01 to 1992-12-31]
  1993: ✓ COMPLETE [1993-01-01 to 1993-12-31]
  1994: ✓ COMPLETE [1994-01-01 to 1994-12-31]
  1995: ✗ 363/365 (99.5%) [1995-01-01 to 1995-12-31]
  1996: ✓ COMPLETE [1996-01-01 to 1996-12-31]
  1997: ✗ 363/365 (99.5%) [1997-01-01 to 1997-12-31]
  1998: ✓ COMPLETE [1998-01-01 to 1998-12-31]
  1999: ✓ COMPLETE [1999-01-01 

# PNNL

In [6]:
print("Scanning PNNL data for coverage...")
print("="*80)

coverage_by_year = {}
all_years = []

# Find all year directories in PNNL/historical
year_folders = [d for d in os.listdir(PNNL_DIR) if os.path.isdir(os.path.join(PNNL_DIR, d)) and d.isdigit()]
year_folders = sorted([int(y) for y in year_folders])

for year in year_folders:
    year_path = os.path.join(PNNL_DIR, str(year))
    nc_file = os.path.join(year_path, f"PNNL_WRF.HIST.CTRL.hourly.PREC_ACC_NC.{year}.nc")

    print(f"\nYear: {year}")

    if not os.path.exists(nc_file):
        print(f"  ✗ File not found: {nc_file}")
        continue

    try:
        ds = xr.open_dataset(nc_file)
        hourly_dates = pd.to_datetime(ds.time.values)

        # Expected dates for the year (hourly)
        expected_dates = pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31, 23), freq='h')

        available_count = len(hourly_dates)
        expected_count = len(expected_dates)
        coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

        # Find missing dates (by day, not hour)
        available_days = set(d.date() for d in hourly_dates)
        expected_days = set(pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31), freq='D').date)
        missing_days = sorted(expected_days - available_days)

        coverage_by_year[year] = {
            'available_hours': available_count,
            'expected_hours': expected_count,
            'coverage_pct': coverage_pct,
            'first_date': hourly_dates.min(),
            'last_date': hourly_dates.max(),
            'missing_days': missing_days,
            'available_days': len(available_days),
            'expected_days': len(expected_days)
        }

        all_years.append(year)

        if coverage_pct == 100:
            status = "✓ COMPLETE"
        else:
            status = f"✗ {available_count}/{expected_count} hours ({coverage_pct:.1f}%)"

        print(f"  {status}")
        print(f"  Range: {hourly_dates.min().strftime('%Y-%m-%d %H:%M')} to {hourly_dates.max().strftime('%Y-%m-%d %H:%M')}")
        print(f"  Days: {len(available_days)}/{len(expected_days)} ({len(available_days)/len(expected_days)*100:.1f}%)")

        ds.close()

    except Exception as e:
        print(f"  Error reading file: {e}")

all_years.sort()

# Print detailed summary
print("\n" + "="*80)
print("PNNL COVERAGE SUMMARY")
print("="*80)

complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

print(f"Complete coverage (100%): {complete} years")
print(f"Partial coverage (1-99%): {partial} years")
print(f"No coverage (0%): {missing} years")
print(f"\nYears available: {all_years[0]} to {all_years[-1]} ({len(all_years)} years total)")

# Show years with incomplete coverage
incomplete = [(y, coverage_by_year[y]) for y in all_years if coverage_by_year[y]['coverage_pct'] < 100]
if incomplete:
    print(f"\n" + "="*80)
    print("YEARS WITH INCOMPLETE COVERAGE")
    print("="*80)
    for year, data in incomplete:
        print(f"\n{year}: {data['coverage_pct']:.1f}% ({data['available_hours']}/{data['expected_hours']} hours)")
        print(f"  Days: {data['available_days']}/{data['expected_days']}")
        if len(data['missing_days']) <= 20:
            print(f"  Missing days ({len(data['missing_days'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_days']])}")
        else:
            missing_by_month = {}
            for d in data['missing_days']:
                key = d.strftime('%Y-%m')
                if key not in missing_by_month:
                    missing_by_month[key] = []
                missing_by_month[key].append(d)

            print(f"  Missing days ({len(data['missing_days'])}): Gaps by month:")
            for month_key in sorted(missing_by_month.keys()):
                days_in_month = missing_by_month[month_key]
                print(f"    {month_key}: {len(days_in_month)} missing days")

Scanning PNNL data for coverage...

Year: 1981
  ✓ COMPLETE
  Range: 1981-01-01 00:00 to 1981-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1982
  ✓ COMPLETE
  Range: 1982-01-01 00:00 to 1982-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1983
  ✓ COMPLETE
  Range: 1983-01-01 00:00 to 1983-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1984
  ✓ COMPLETE
  Range: 1984-01-01 00:00 to 1984-12-31 23:00
  Days: 366/366 (100.0%)

Year: 1985
  ✓ COMPLETE
  Range: 1985-01-01 00:00 to 1985-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1986
  ✓ COMPLETE
  Range: 1986-01-01 00:00 to 1986-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1987
  ✓ COMPLETE
  Range: 1987-01-01 00:00 to 1987-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1988
  ✓ COMPLETE
  Range: 1988-01-01 00:00 to 1988-12-31 23:00
  Days: 366/366 (100.0%)

Year: 1989
  ✓ COMPLETE
  Range: 1989-01-01 00:00 to 1989-12-31 23:00
  Days: 365/365 (100.0%)

Year: 1990
  ✓ COMPLETE
  Range: 1990-01-01 00:00 to 1990-12-31 23:00
  Days: 365/365 (100.0%)

Year

# Daymet

I downloaded this file from the ORNL side, adn it happens to be that it is aggregated to 4 km cells. I want the original Daymet (1 km grid cells)

In [26]:
example_daymet = xr.open_dataset("/data0/hernanqd/plots_code/skagit_basin_de/cumulative_precipitation_plot/exploring/DaymetV4_VIC4_prcp_1980.nc")
example_daymet

<xarray.Dataset> Size: 1GB
Dimensions:  (time: 366, lat: 697, lon: 1405)
Coordinates:
  * time     (time) datetime64[ns] 3kB 1980-01-01T12:00:00 ... 1980-12-31T12:...
  * lat      (lat) float64 6kB 24.0 24.04 24.08 24.12 ... 52.87 52.92 52.96 53.0
  * lon      (lon) float64 11kB -125.0 -125.0 -124.9 ... -66.58 -66.54 -66.5
Data variables:
    prcp     (time, lat, lon) float32 1GB ...
Attributes:
    title:                    SECURE Water Act 9505V3 Assessment
    experiment:               DaymetV4_VIC5
    domain:                   Conterminous US (CONUS)
    Forcing:                  DaymetV4, aggregated to 4km ORNL CONUS-VIC4 grids
    VIC_parameters:           ORNL CONUS-VIC4 v20200704D
    computational_resources:  Oak Ridge Leadership Computing Facility at the ...
    institution:              Oak Ridge National Laboratory, Oak Ridge, TN, USA
    contact:                  Shih-Chieh Kao (kaos@ornl.gov)
    creation_date:            19-Jul-2023 14:35:47
    Conventions:              CF-1.8

In [27]:
# Get spatial resolution
lat = example_daymet.lat.values
lon = example_daymet.lon.values

# Calculate resolution (difference between consecutive points)
lat_resolution = np.abs(lat[1] - lat[0])
lon_resolution = np.abs(lon[1] - lon[0])

print("Daymet Spatial Resolution")
print("="*80)
print(f"Latitude resolution: {lat_resolution}° ({lat_resolution * 111:.2f} km)")
print(f"Longitude resolution: {lon_resolution}° ({lon_resolution * 111:.2f} km)")
print(f"\nGrid dimensions:")
print(f"  Latitude range: {lat.min()}° to {lat.max()}° ({len(lat)} points)")
print(f"  Longitude range: {lon.min()}° to {lon.max()}° ({len(lon)} points)")
print(f"  Total grid cells: {len(lat) * len(lon):,}")

# Show full dataset info
print(f"\n{example_daymet}")

example_daymet.close()

Daymet Spatial Resolution
Latitude resolution: 0.04166666666666785° (4.63 km)
Longitude resolution: 0.041666666666671404° (4.63 km)

Grid dimensions:
  Latitude range: 23.999999999999° to 52.999999999999° (697 points)
  Longitude range: -125.00000000001° to -66.50000000001° (1405 points)
  Total grid cells: 979,285

<xarray.Dataset> Size: 1GB
Dimensions:  (time: 366, lat: 697, lon: 1405)
Coordinates:
  * time     (time) datetime64[ns] 3kB 1980-01-01T12:00:00 ... 1980-12-31T12:...
  * lat      (lat) float64 6kB 24.0 24.04 24.08 24.12 ... 52.87 52.92 52.96 53.0
  * lon      (lon) float64 11kB -125.0 -125.0 -124.9 ... -66.58 -66.54 -66.5
Data variables:
    prcp     (time, lat, lon) float32 1GB ...
Attributes:
    title:                    SECURE Water Act 9505V3 Assessment
    experiment:               DaymetV4_VIC5
    domain:                   Conterminous US (CONUS)
    Forcing:                  DaymetV4, aggregated to 4km ORNL CONUS-VIC4 grids
    VIC_parameters:           ORNL CONUS

In [21]:
VAULT_DIR = "/data0/skagit_met/data_transfer/data"

def get_ornl_zarr(year):
    """Get the Daymet/ORNL zarr path for a given year"""
    if 1981 <= year <= 2011:
        p = os.path.join(VAULT_DIR, "climate_sets/1981_2011_ORNL_data.zarr")
        if os.path.exists(p):
            return p
    p1 = os.path.join(VAULT_DIR, f"ornl/{year}_{year}_ref_DaymetV4_ORNL_data.zarr")
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(VAULT_DIR, f"DaymetV4/{year}_{year}_ORNL_data.zarr")
    if os.path.exists(p2):
        return p2
    return None

print("Scanning Daymet data for coverage...")
print("="*80)
print("Note: Daymet may be stored in multiple locations:")
print("  - climate_sets/1981_2011_ORNL_data.zarr (consolidated 1981-2011)")
print("  - ornl/{year}_{year}_ref_DaymetV4_ORNL_data.zarr")
print("  - DaymetV4/{year}_{year}_ORNL_data.zarr")
print("="*80)

coverage_by_year = {}
all_years = []
consolidated_file_used = False

# Scan years from 1980 to current
for year in range(1980, 2026):
    daymet_path = get_ornl_zarr(year)

    if daymet_path is None:
        continue

    print(f"\nYear: {year}")

    # Check if this is the consolidated file
    if "climate_sets" in daymet_path and not consolidated_file_used:
        print(f"  Using consolidated file: climate_sets/1981_2011_ORNL_data.zarr")
        consolidated_file_used = True
    else:
        # Extract which directory this file is in
        if "ornl" in daymet_path:
            location = "ornl/"
        elif "DaymetV4" in daymet_path:
            location = "DaymetV4/"
        else:
            location = "climate_sets/"
        print(f"  Path: {location}{os.path.basename(daymet_path)}")

    try:
        is_cons = (1981 <= year <= 2011)
        ds = xr.open_zarr(daymet_path, consolidated=is_cons)

        # Handle different time dimension names
        if 'day' in ds.dims:
            ds = ds.rename({'day': 'time'})

        dates = pd.to_datetime(ds.time.values).normalize()

        # Expected dates for the year
        expected_dates = pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31), freq='D')

        available_count = len(dates)
        expected_count = len(expected_dates)
        coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

        # Find missing dates
        missing_dates = [d for d in expected_dates if d not in dates.values]

        coverage_by_year[year] = {
            'available': available_count,
            'expected': expected_count,
            'coverage_pct': coverage_pct,
            'first_date': dates.min(),
            'last_date': dates.max(),
            'missing_dates': missing_dates
        }

        all_years.append(year)

        if coverage_pct == 100:
            status = "✓ COMPLETE"
        else:
            status = f"✗ {available_count}/{expected_count} ({coverage_pct:.1f}%)"

        print(f"  {status}")
        print(f"  Range: {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")

        if not is_cons:
            ds.close()

    except Exception as e:
        print(f"  Error reading file: {e}")

all_years.sort()

# Print detailed summary
print("\n" + "="*80)
print("DAYMET COVERAGE SUMMARY")
print("="*80)

complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

print(f"Complete coverage (100%): {complete} years")
print(f"Partial coverage (1-99%): {partial} years")
print(f"No coverage (0%): {missing} years")

if all_years:
    print(f"\nYears available: {all_years[0]} to {all_years[-1]} ({len(all_years)} years total)")

# Show years with incomplete coverage
incomplete = [(y, coverage_by_year[y]) for y in all_years if coverage_by_year[y]['coverage_pct'] < 100]
if incomplete:
    print(f"\n" + "="*80)
    print("YEARS WITH INCOMPLETE COVERAGE")
    print("="*80)
    for year, data in incomplete:
        print(f"\nYear {year}: {data['coverage_pct']:.1f}% ({data['available']}/{data['expected']} dates)")
        if len(data['missing_dates']) <= 20:
            print(f"  Missing dates ({len(data['missing_dates'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_dates']])}")
        else:
            missing_by_month = {}
            for d in data['missing_dates']:
                key = d.strftime('%Y-%m')
                if key not in missing_by_month:
                    missing_by_month[key] = []
                missing_by_month[key].append(d)

            print(f"  Missing dates ({len(data['missing_dates'])}): Gaps by month:")
            for month_key in sorted(missing_by_month.keys()):
                dates_in_month = missing_by_month[month_key]
                print(f"    {month_key}: {len(dates_in_month)} missing days")

Scanning Daymet data for coverage...
Note: Daymet may be stored in multiple locations:
  - climate_sets/1981_2011_ORNL_data.zarr (consolidated 1981-2011)
  - ornl/{year}_{year}_ref_DaymetV4_ORNL_data.zarr
  - DaymetV4/{year}_{year}_ORNL_data.zarr

Year: 1981
  Using consolidated file: climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1982
  Path: climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1983
  Path: climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1984
  Path: climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/366 (3093.4%)
  Range: 1981-01-01 to 2011-12-31

Year: 1985
  Path: climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1986
  Path: climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1987
  Path: climate_sets/1981_

In [10]:
def get_ornl_zarr(year):
    """Get the Daymet/ORNL zarr path for a given year"""
    if 1981 <= year <= 2011:
        p = os.path.join(VAULT_DIR, "climate_sets/1981_2011_ORNL_data.zarr")
        if os.path.exists(p):
            return p
    p1 = os.path.join(VAULT_DIR, f"ornl/{year}_{year}_ref_DaymetV4_ORNL_data.zarr")
    if os.path.exists(p1):
        return p1
    p2 = os.path.join(VAULT_DIR, f"DaymetV4/{year}_{year}_ORNL_data.zarr")
    if os.path.exists(p2):
        return p2
    return None

print("Scanning Daymet data for coverage...")
print("="*80)

coverage_by_year = {}
all_years = []

# Scan years from 1980 to current
for year in range(1980, 2026):
    daymet_path = get_ornl_zarr(year)

    if daymet_path is None:
        continue

    print(f"\nYear: {year}")
    print(f"  Path: {daymet_path}")

    try:
        is_cons = (1981 <= year <= 2011)
        ds = xr.open_zarr(daymet_path, consolidated=is_cons)

        # Handle different time dimension names
        if 'day' in ds.dims:
            ds = ds.rename({'day': 'time'})

        dates = pd.to_datetime(ds.time.values).normalize()

        # Expected dates for the year
        expected_dates = pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31), freq='D')

        available_count = len(dates)
        expected_count = len(expected_dates)
        coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

        # Find missing dates
        missing_dates = [d for d in expected_dates if d not in dates.values]

        coverage_by_year[year] = {
            'zarr_path': daymet_path,
            'available': available_count,
            'expected': expected_count,
            'coverage_pct': coverage_pct,
            'first_date': dates.min(),
            'last_date': dates.max(),
            'missing_dates': missing_dates
        }

        all_years.append(year)

        if coverage_pct == 100:
            status = "✓ COMPLETE"
        else:
            status = f"✗ {available_count}/{expected_count} ({coverage_pct:.1f}%)"

        print(f"  {status}")
        print(f"  Range: {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")

        if not is_cons:
            ds.close()

    except Exception as e:
        print(f"  Error reading file: {e}")

all_years.sort()

# Print detailed summary
print("\n" + "="*80)
print("DAYMET COVERAGE SUMMARY")
print("="*80)

complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

print(f"Complete coverage (100%): {complete} years")
print(f"Partial coverage (1-99%): {partial} years")
print(f"No coverage (0%): {missing} years")

if all_years:
    print(f"\nYears available: {all_years[0]} to {all_years[-1]} ({len(all_years)} years total)")

# Show years with incomplete coverage
incomplete = [(y, coverage_by_year[y]) for y in all_years if coverage_by_year[y]['coverage_pct'] < 100]
if incomplete:
    print(f"\n" + "="*80)
    print("YEARS WITH INCOMPLETE COVERAGE")
    print("="*80)
    for year, data in incomplete:
        print(f"\n{year}: {data['coverage_pct']:.1f}% ({data['available']}/{data['expected']} dates)")
        print(f"  Zarr: {data['zarr_path']}")
        if len(data['missing_dates']) <= 20:
            print(f"  Missing dates ({len(data['missing_dates'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_dates']])}")
        else:
            missing_by_month = {}
            for d in data['missing_dates']:
                key = d.strftime('%Y-%m')
                if key not in missing_by_month:
                    missing_by_month[key] = []
                missing_by_month[key].append(d)

            print(f"  Missing dates ({len(data['missing_dates'])}): Gaps by month:")
            for month_key in sorted(missing_by_month.keys()):
                dates_in_month = missing_by_month[month_key]
                print(f"    {month_key}: {len(dates_in_month)} missing days")

Scanning Daymet data for coverage...

Year: 1981
  Path: /data0/skagit_met/data_transfer/data/climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1982
  Path: /data0/skagit_met/data_transfer/data/climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1983
  Path: /data0/skagit_met/data_transfer/data/climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1984
  Path: /data0/skagit_met/data_transfer/data/climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/366 (3093.4%)
  Range: 1981-01-01 to 2011-12-31

Year: 1985
  Path: /data0/skagit_met/data_transfer/data/climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1986
  Path: /data0/skagit_met/data_transfer/data/climate_sets/1981_2011_ORNL_data.zarr
  ✗ 11322/365 (3101.9%)
  Range: 1981-01-01 to 2011-12-31

Year: 1987
  Path: /data0/skagit_met/data_tr

# UCLA

In [17]:
dataset = xr.open_dataset(UCLA_DIR + "/prec.daily.era5.d02.2019.nc")
dataset

<xarray.Dataset> Size: 134MB
Dimensions:  (day: 366, lat2d: 340, lon2d: 270)
Coordinates:
  * day      (day) datetime64[ns] 3kB 2019-09-01 2019-09-02 ... 2020-08-31
Dimensions without coordinates: lat2d, lon2d
Data variables:
    prec     (day, lat2d, lon2d) float32 134MB ...
Attributes: (12/152)
    TITLE:                            OUTPUT FROM WRF V4.1.3 MODEL
    START_DATE:                      2020-05-07_00:00:00
    SIMULATION_START_DATE:           2019-08-01_00:00:00
    WEST_EAST_GRID_DIMENSION:        271
    SOUTH_NORTH_GRID_DIMENSION:      341
    BOTTOM_TOP_GRID_DIMENSION:       40
    ...                              ...
    ISOILWATER:                      14
    HYBRID_OPT:                      0
    ETAC:                            0.0
    Conventions:                     CF-1.7
    institution:                     UCLA Center for Climate Science
    source:                          https://dept.atmos.ucla.edu/alexhall/dow...

In [16]:
print("Scanning UCLA data for coverage (calendar years)...")
print("="*80)
print("Note: File YYYY contains Sep 1 YYYY to Aug 31 YYYY+1")
print("      So calendar year 2003: Jan-Aug from file 2002, Sep-Dec from file 2003")
print("="*80)

coverage_by_year = {}
all_years = []

# Find all available year files
nc_files = [f for f in os.listdir(UCLA_DIR) if f.endswith('.nc')]
available_file_years = set()
for nc_file in nc_files:
    try:
        year = int(nc_file.split('.')[-2])
        available_file_years.add(year)
    except:
        continue

available_file_years = sorted(available_file_years)

# Check coverage for calendar years
for calendar_year in range(min(available_file_years), max(available_file_years)):
    print(f"\nCalendar Year: {calendar_year}")

    # For calendar year YYYY:
    # - Jan 1 to Aug 31: from file (YYYY-1)
    # - Sep 1 to Dec 31: from file YYYY
    
    all_dates_in_year = []
    files_used = []

    # Get Jan-Aug from previous year's file
    prev_file_year = calendar_year - 1
    prev_nc_path = os.path.join(UCLA_DIR, f"prec.daily.era5.d02.{prev_file_year}.nc")

    if os.path.exists(prev_nc_path):
        try:
            ds = xr.open_dataset(prev_nc_path)
            u_var = "prec" if "prec" in ds.data_vars else "pr"
            if 'day' in ds.dims:
                ds = ds.rename({'day': 'time'})
            dates = pd.to_datetime(ds.time.values).normalize()
            
            # Extract only Jan-Aug of the calendar year
            jan_aug_dates = [d for d in dates if d.year == calendar_year and d.month <= 8]
            all_dates_in_year.extend(jan_aug_dates)
            files_used.append(f"{prev_file_year} (Jan-Aug)")
            
            ds.close()
        except Exception as e:
            print(f"  Error reading file {prev_file_year}: {e}")
    else:
        print(f"  ⚠ File for {prev_file_year} not found (needed for Jan-Aug)")

    # Get Sep-Dec from current year's file
    curr_nc_path = os.path.join(UCLA_DIR, f"prec.daily.era5.d02.{calendar_year}.nc")

    if os.path.exists(curr_nc_path):
        try:
            ds = xr.open_dataset(curr_nc_path)
            u_var = "prec" if "prec" in ds.data_vars else "pr"
            if 'day' in ds.dims:
                ds = ds.rename({'day': 'time'})
            dates = pd.to_datetime(ds.time.values).normalize()
            
            # Extract only Sep-Dec of the calendar year
            sep_dec_dates = [d for d in dates if d.year == calendar_year and d.month >= 9]
            all_dates_in_year.extend(sep_dec_dates)
            files_used.append(f"{calendar_year} (Sep-Dec)")
            
            ds.close()
        except Exception as e:
            print(f"  Error reading file {calendar_year}: {e}")
    else:
        print(f"  ⚠ File for {calendar_year} not found (needed for Sep-Dec)")

    if not all_dates_in_year:
        print(f"  ✗ No data found")
        continue

    all_dates_in_year = sorted(set(all_dates_in_year))

    # Expected dates for calendar year
    expected_dates = pd.date_range(start=pd.Timestamp(calendar_year, 1, 1), end=pd.Timestamp(calendar_year, 12, 31), freq='D')

    available_count = len(all_dates_in_year)
    expected_count = len(expected_dates)
    coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

    # Find missing dates
    missing_dates = [d for d in expected_dates if d not in all_dates_in_year]

    coverage_by_year[calendar_year] = {
        'available': available_count,
        'expected': expected_count,
        'coverage_pct': coverage_pct,
        'first_date': min(all_dates_in_year),
        'last_date': max(all_dates_in_year),
        'missing_dates': missing_dates,
        'files_used': files_used
    }

    all_years.append(calendar_year)

    if coverage_pct == 100:
        status = "✓ COMPLETE"
    else:
        status = f"✗ {available_count}/{expected_count} ({coverage_pct:.1f}%)"

    print(f"  {status}")
    print(f"  Files: {', '.join(files_used)}")
    print(f"  Range: {min(all_dates_in_year).strftime('%Y-%m-%d')} to {max(all_dates_in_year).strftime('%Y-%m-%d')}")

# Print detailed summary
print("\n" + "="*80)
print("UCLA COVERAGE SUMMARY (Calendar Years)")
print("="*80)

complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

print(f"Complete coverage (100%): {complete} years")
print(f"Partial coverage (1-99%): {partial} years")
print(f"No coverage (0%): {missing} years")

if all_years:
    print(f"\nYears available: {all_years[0]} to {all_years[-1]} ({len(all_years)} years total)")

# Show years with incomplete coverage
incomplete = [(y, coverage_by_year[y]) for y in all_years if coverage_by_year[y]['coverage_pct'] < 100]
if incomplete:
    print(f"\n" + "="*80)
    print("YEARS WITH INCOMPLETE COVERAGE")
    print("="*80)
    for year, data in incomplete:
        print(f"\nYear {year}: {data['coverage_pct']:.1f}% ({data['available']}/{data['expected']} dates)")
        print(f"  Files used: {', '.join(data['files_used'])}")
        if len(data['missing_dates']) <= 20:
            print(f"  Missing dates ({len(data['missing_dates'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_dates']])}")
        else:
            missing_by_month = {}
            for d in data['missing_dates']:
                key = d.strftime('%Y-%m')
                if key not in missing_by_month:
                    missing_by_month[key] = []
                missing_by_month[key].append(d)

            print(f"  Missing dates ({len(data['missing_dates'])}): Gaps by month:")
            for month_key in sorted(missing_by_month.keys()):
                dates_in_month = missing_by_month[month_key]
                print(f"    {month_key}: {len(dates_in_month)} missing days")

Scanning UCLA data for coverage (calendar years)...
Note: File YYYY contains Sep 1 YYYY to Aug 31 YYYY+1
      So calendar year 2003: Jan-Aug from file 2002, Sep-Dec from file 2003

Calendar Year: 1950
  ⚠ File for 1949 not found (needed for Jan-Aug)
  ✗ 122/365 (33.4%)
  Files: 1950 (Sep-Dec)
  Range: 1950-09-01 to 1950-12-31

Calendar Year: 1951
  ✓ COMPLETE
  Files: 1950 (Jan-Aug), 1951 (Sep-Dec)
  Range: 1951-01-01 to 1951-12-31

Calendar Year: 1952
  ✓ COMPLETE
  Files: 1951 (Jan-Aug), 1952 (Sep-Dec)
  Range: 1952-01-01 to 1952-12-31

Calendar Year: 1953
  ✓ COMPLETE
  Files: 1952 (Jan-Aug), 1953 (Sep-Dec)
  Range: 1953-01-01 to 1953-12-31

Calendar Year: 1954
  ✓ COMPLETE
  Files: 1953 (Jan-Aug), 1954 (Sep-Dec)
  Range: 1954-01-01 to 1954-12-31

Calendar Year: 1955
  ✓ COMPLETE
  Files: 1954 (Jan-Aug), 1955 (Sep-Dec)
  Range: 1955-01-01 to 1955-12-31

Calendar Year: 1956
  ✓ COMPLETE
  Files: 1955 (Jan-Aug), 1956 (Sep-Dec)
  Range: 1956-01-01 to 1956-12-31

Calendar Year: 1957
  

# Gridmet

In [19]:
print("Scanning GridMET data for coverage...")
print("="*80)

coverage_by_year = {}
all_years = []

# Find all zarr files in gridmet directory
if not os.path.exists(GRIDMET_DIR):
    print(f"GridMET directory not found: {GRIDMET_DIR}")
else:
    zarr_files = [f for f in os.listdir(GRIDMET_DIR) if f.endswith('.zarr')]
    zarr_files.sort()

    for zarr_file in zarr_files:
        zarr_path = os.path.join(GRIDMET_DIR, zarr_file)

        # Extract year from filename (e.g., "2019_daily_4km_gridMET_data.zarr" -> 2019)
        try:
            year = int(zarr_file.split('_')[0])
        except:
            continue

        print(f"\nYear: {year}")

        try:
            ds = xr.open_zarr(zarr_path, consolidated=False)

            # Handle different time dimension names
            if 'day' in ds.dims:
                ds = ds.rename({'day': 'time'})

            # Handle different variable names
            var = 'prcp' if 'prcp' in ds.data_vars else 'precipitation_amount'

            dates = pd.to_datetime(ds.time.values).normalize()

            # Expected dates for the year
            expected_dates = pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31), freq='D')

            available_count = len(dates)
            expected_count = len(expected_dates)
            coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

            # Find missing dates
            missing_dates = [d for d in expected_dates if d not in dates.values]

            coverage_by_year[year] = {
                'variable': var,
                'available': available_count,
                'expected': expected_count,
                'coverage_pct': coverage_pct,
                'first_date': dates.min(),
                'last_date': dates.max(),
                'missing_dates': missing_dates
            }

            all_years.append(year)

            if coverage_pct == 100:
                status = "✓ COMPLETE"
            else:
                status = f"✗ {available_count}/{expected_count} ({coverage_pct:.1f}%)"

            print(f"  {status} (variable: {var})")
            print(f"  Range: {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")

            ds.close()

        except Exception as e:
            print(f"  Error reading file: {e}")

all_years.sort()

# Print detailed summary
print("\n" + "="*80)
print("GRIDMET COVERAGE SUMMARY")
print("="*80)

complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

print(f"Complete coverage (100%): {complete} years")
print(f"Partial coverage (1-99%): {partial} years")
print(f"No coverage (0%): {missing} years")

if all_years:
    print(f"\nYears available: {all_years[0]} to {all_years[-1]} ({len(all_years)} years total)")

# Show years with incomplete coverage
incomplete = [(y, coverage_by_year[y]) for y in all_years if coverage_by_year[y]['coverage_pct'] < 100]
if incomplete:
    print(f"\n" + "="*80)
    print("YEARS WITH INCOMPLETE COVERAGE")
    print("="*80)
    for year, data in incomplete:
        print(f"\nYear {year}: {data['coverage_pct']:.1f}% ({data['available']}/{data['expected']} dates)")
        if len(data['missing_dates']) <= 20:
            print(f"  Missing dates ({len(data['missing_dates'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_dates']])}")
        else:
            missing_by_month = {}
            for d in data['missing_dates']:
                key = d.strftime('%Y-%m')
                if key not in missing_by_month:
                    missing_by_month[key] = []
                missing_by_month[key].append(d)

            print(f"  Missing dates ({len(data['missing_dates'])}): Gaps by month:")
            for month_key in sorted(missing_by_month.keys()):
                dates_in_month = missing_by_month[month_key]
                print(f"    {month_key}: {len(dates_in_month)} missing days")

Scanning GridMET data for coverage...

Year: 1981
  ✓ COMPLETE (variable: prcp)
  Range: 1981-01-01 to 1981-12-31

Year: 1982
  ✓ COMPLETE (variable: prcp)
  Range: 1982-01-01 to 1982-12-31

Year: 1983
  ✓ COMPLETE (variable: prcp)
  Range: 1983-01-01 to 1983-12-31

Year: 1984
  ✓ COMPLETE (variable: prcp)
  Range: 1984-01-01 to 1984-12-31

Year: 1985
  ✓ COMPLETE (variable: prcp)
  Range: 1985-01-01 to 1985-12-31

Year: 1986
  ✓ COMPLETE (variable: prcp)
  Range: 1986-01-01 to 1986-12-31

Year: 1987
  ✓ COMPLETE (variable: prcp)
  Range: 1987-01-01 to 1987-12-31

Year: 1988
  ✓ COMPLETE (variable: prcp)
  Range: 1988-01-01 to 1988-12-31

Year: 1989
  ✓ COMPLETE (variable: prcp)
  Range: 1989-01-01 to 1989-12-31

Year: 1990
  ✓ COMPLETE (variable: prcp)
  Range: 1990-01-01 to 1990-12-31

Year: 1991
  ✓ COMPLETE (variable: prcp)
  Range: 1991-01-01 to 1991-12-31

Year: 1992
  ✓ COMPLETE (variable: prcp)
  Range: 1992-01-01 to 1992-12-31

Year: 1993
  ✓ COMPLETE (variable: prcp)
  Range:

# CONUS

In [20]:
BASE_DIR = "/data0/hernanqd/plots_code/skagit_basin_de"
CONUS_PATH = os.path.join(BASE_DIR, "data/weather_data/conus404_skagit_precip_daily_full.zarr")

print("Scanning CONUS404 data for coverage...")
print("="*80)
print("Note: CONUS404 is a single zarr file covering all years")
print("="*80)

if not os.path.exists(CONUS_PATH):
    print(f"CONUS404 file not found: {CONUS_PATH}")
else:
    try:
        ds = xr.open_zarr(CONUS_PATH, consolidated=False)

        dates = pd.to_datetime(ds.time.values).normalize()

        print(f"\nFile: conus404_skagit_precip_daily_full.zarr")
        print(f"Variable: precip_daily")
        print(f"Total days available: {len(dates)}")

        # Find year range
        years_covered = sorted(set(d.year for d in dates))
        year_start = years_covered[0]
        year_end = years_covered[-1]

        print(f"Year range: {year_start} to {year_end}")

        # Check coverage by year
        print(f"\nCoverage by year:")
        print("-" * 80)

        coverage_by_year = {}

        for year in range(year_start, year_end + 1):
            year_dates = [d for d in dates if d.year == year]
            expected_dates = pd.date_range(start=pd.Timestamp(year, 1, 1), end=pd.Timestamp(year, 12, 31), freq='D')

            available_count = len(year_dates)
            expected_count = len(expected_dates)
            coverage_pct = (available_count / expected_count * 100) if expected_count > 0 else 0

            missing_dates = [d for d in expected_dates if d not in dates.values]

            coverage_by_year[year] = {
                'available': available_count,
                'expected': expected_count,
                'coverage_pct': coverage_pct,
                'missing_dates': missing_dates
            }

            if coverage_pct == 100:
                status = "✓"
            elif coverage_pct > 0:
                status = f"⚠ {available_count}/{expected_count}"
            else:
                status = "✗"

            print(f"  {year}: {status} ({coverage_pct:.1f}%)")

        # Print detailed summary
        print("\n" + "="*80)
        print("CONUS404 COVERAGE SUMMARY")
        print("="*80)

        complete = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 100)
        partial = sum(1 for d in coverage_by_year.values() if 0 < d['coverage_pct'] < 100)
        missing = sum(1 for d in coverage_by_year.values() if d['coverage_pct'] == 0)

        print(f"Complete coverage (100%): {complete} years")
        print(f"Partial coverage (1-99%): {partial} years")
        print(f"No coverage (0%): {missing} years")

        print(f"\nDate range: {dates.min().strftime('%Y-%m-%d')} to {dates.max().strftime('%Y-%m-%d')}")
        print(f"Total days: {len(dates)}")

        # Show years with incomplete coverage
        incomplete = [(y, coverage_by_year[y]) for y in sorted(coverage_by_year.keys()) if coverage_by_year[y]['coverage_pct'] < 100]
        if incomplete:
            print(f"\n" + "="*80)
            print("YEARS WITH INCOMPLETE COVERAGE")
            print("="*80)
            for year, data in incomplete:
                print(f"\nYear {year}: {data['coverage_pct']:.1f}% ({data['available']}/{data['expected']} dates)")
                if len(data['missing_dates']) <= 20:
                    print(f"  Missing dates ({len(data['missing_dates'])}): {', '.join([d.strftime('%Y-%m-%d') for d in data['missing_dates']])}")
                else:
                    missing_by_month = {}
                    for d in data['missing_dates']:
                        key = d.strftime('%Y-%m')
                        if key not in missing_by_month:
                            missing_by_month[key] = []
                        missing_by_month[key].append(d)

                    print(f"  Missing dates ({len(data['missing_dates'])}): Gaps by month:")
                    for month_key in sorted(missing_by_month.keys()):
                        dates_in_month = missing_by_month[month_key]
                        print(f"    {month_key}: {len(dates_in_month)} missing days")

        ds.close()

    except Exception as e:
        print(f"Error reading CONUS404 file: {e}")

Scanning CONUS404 data for coverage...
Note: CONUS404 is a single zarr file covering all years

File: conus404_skagit_precip_daily_full.zarr
Variable: precip_daily
Total days available: 15979
Year range: 1981 to 2024

Coverage by year:
--------------------------------------------------------------------------------
  1981: ✓ (100.0%)
  1982: ✓ (100.0%)
  1983: ✓ (100.0%)
  1984: ✓ (100.0%)
  1985: ✓ (100.0%)
  1986: ✓ (100.0%)
  1987: ✓ (100.0%)
  1988: ✓ (100.0%)
  1989: ✓ (100.0%)
  1990: ✓ (100.0%)
  1991: ✓ (100.0%)
  1992: ✓ (100.0%)
  1993: ✓ (100.0%)
  1994: ✓ (100.0%)
  1995: ✓ (100.0%)
  1996: ✓ (100.0%)
  1997: ✓ (100.0%)
  1998: ✓ (100.0%)
  1999: ✓ (100.0%)
  2000: ✓ (100.0%)
  2001: ✓ (100.0%)
  2002: ✓ (100.0%)
  2003: ✓ (100.0%)
  2004: ✓ (100.0%)
  2005: ✓ (100.0%)
  2006: ✓ (100.0%)
  2007: ✓ (100.0%)
  2008: ✓ (100.0%)
  2009: ✓ (100.0%)
  2010: ✓ (100.0%)
  2011: ✓ (100.0%)
  2012: ✓ (100.0%)
  2013: ✓ (100.0%)
  2014: ✓ (100.0%)
  2015: ✓ (100.0%)
  2016: ✓ (100.0%)